# GemCol Evaluation: Phase 5 (ColBERTv2 Reranking)
This notebook applies ColBERT's Late Interaction MaxSim scoring to rerank the top 100 candidate documents retrieved from the RRF Fusion phase. This acts as a highly accurate cross-encoder step, drastically boosting nDCG.

In [ ]:
import sys
import os
import json
import torch
import torch.nn as nn
from tqdm import tqdm
from transformers import BertPreTrainedModel, BertModel, AutoTokenizer, BertConfig
from huggingface_hub import hf_hub_download
import safetensors.torch

sys.path.insert(0, '/workspace/gemcol_evaluation')
from utils import load_msmarco_dev, save_run_json, evaluate_run, print_metrics, LatencyProfiler, ExperimentTracker

print("Loading dataset...")
queries, qrels, corpus = load_msmarco_dev()

print("Loading RRF run candidates...")
with open("/workspace/results/rrf_run.json", "r") as f:
    rrf_run = json.load(f)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class ColBERT(nn.Module):
    def __init__(self):
        super().__init__()
        config = BertConfig.from_pretrained("colbert-ir/colbertv2.0")
        self.bert = BertModel(config)
        self.linear = nn.Linear(config.hidden_size, 128, bias=False)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        embeddings = self.linear(outputs.last_hidden_state)
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=-1)
        return embeddings

print("Loading ColBERTv2 model natively...")
tokenizer = AutoTokenizer.from_pretrained("colbert-ir/colbertv2.0")
model = ColBERT()
model_path = hf_hub_download("colbert-ir/colbertv2.0", "model.safetensors")
state_dict = safetensors.torch.load_file(model_path)
model.load_state_dict(state_dict, strict=False)
model = model.to(device)
model.eval()
print("✅ Model loaded!")

In [ ]:
def encode_query(query_text):
    # ColBERT prepends queries with [unused0]
    q_text = "[unused0] " + query_text
    # ColBERT pads queries to 32 tokens and replaces [PAD] with [MASK] to augment the query representation
    inputs = tokenizer(q_text, return_tensors="pt", max_length=32, truncation=True, padding="max_length").to(device)
    inputs['input_ids'][inputs['input_ids'] == tokenizer.pad_token_id] = tokenizer.mask_token_id
    
    with torch.no_grad():
        Q = model(**inputs)  # Shape: (1, 32, 128)
    return Q

def encode_documents_and_score(Q, doc_texts):
    # ColBERT prepends documents with [unused1]
    d_texts = ["[unused1] " + d for d in doc_texts]
    inputs = tokenizer(d_texts, return_tensors="pt", max_length=150, truncation=True, padding=True).to(device)
    
    with torch.no_grad():
        D = model(**inputs)  # Shape: (batch_size, seq_len, 128)
    
    # --- MaxSim Calculation ---
    # Q: (1, L_q, 128)
    # D: (batch, L_d, 128)
    # Calculate dot product similarities. Result shape: (batch, L_q, L_d)
    sim = torch.einsum("iqd,bjd->bqj", Q, D)
    
    # Mask out the padding tokens in the documents so they don't get selected in the Max
    d_mask = inputs['attention_mask'].unsqueeze(1)  # (batch, 1, L_d)
    sim = sim.masked_fill(d_mask == 0, -10000.0)
    
    # For each query token, find the maximum similarity across all document tokens
    max_scores, _ = torch.max(sim, dim=2)  # (batch, L_q)
    
    # Sum the max scores over the query tokens
    final_scores = torch.sum(max_scores, dim=1)  # (batch,)
    return final_scores


In [ ]:
print("Starting ColBERTv2 Reranking...")
colbert_run = {}
prof = LatencyProfiler()

# Track resume checkpoint to avoid losing progress if interrupted
checkpoint_path = "/workspace/results/colbert_checkpoint.json"
if os.path.exists(checkpoint_path):
    print("Resuming from checkpoint...")
    with open(checkpoint_path, "r") as f:
        colbert_run = json.load(f)

query_ids = list(rrf_run.keys())
for i, qid in enumerate(tqdm(query_ids, desc="Reranking Queries")):
    if qid in colbert_run:
        continue
        
    query_text = queries[qid]
    candidate_doc_ids = rrf_run[qid]
    candidate_texts = [corpus[doc_id] for doc_id in candidate_doc_ids]
    
    with prof.time("colbert_reranking"):
        Q = encode_query(query_text)
        # We encode and score all 100 documents in a single batch!
        scores = encode_documents_and_score(Q, candidate_texts)
    
    # Sort the document IDs by their ColBERT MaxSim scores (descending)
    scores = scores.cpu().numpy()
    sorted_pairs = sorted(zip(candidate_doc_ids, scores), key=lambda x: x[1], reverse=True)
    colbert_run[qid] = [doc_id for doc_id, score in sorted_pairs]
    
    # Save a checkpoint every 100 queries
    if i % 100 == 0:
        with open(checkpoint_path, "w") as f:
            json.dump(colbert_run, f)

save_run_json(colbert_run, "/workspace/results/colbert_run.json")
print("✅ Reranking complete and saved.")

In [ ]:
metrics = evaluate_run(colbert_run, qrels)
print_metrics(metrics, system_name="ColBERTv2 Reranker")

print("\nLatency Profiling:")
prof.print_summary()

tracker = ExperimentTracker("/workspace/results/experiments.json")
tracker.log("ColBERTv2 Reranker", config={"model": "colbert-ir/colbertv2.0", "candidates": 100}, metrics={
    "ndcg@10": metrics["ndcg@10"],
    "mrr@10": metrics["mrr@10"],
    "recall@100": metrics["recall@100"],
    "latency_ms": prof.summary()["colbert_reranking"]["mean_ms"]
})
print("✅ Metrics logged to experiments.json")